<a href="https://colab.research.google.com/github/armandochernandez-ai/Curso-python-slava/blob/main/CUCEA/FRUTAS_HORTALIZAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install selenium
import pandas as pd
import time
import re
import os
from datetime import datetime, timedelta
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
import logging
import sys

# Configuración para Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
    print("✅ Ejecutando en Google Colab")
except:
    IN_COLAB = False
    print("❌ No se detectó Google Colab")

# Configuración de logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

class SNIIMExtractorSelenium:
    def __init__(self):
        self.base_url = "https://www.economia-sniim.gob.mx/nuevo/Home.aspx?opcion=Consultas/MercadosNacionales/PreciosDeMercado/Agricolas/ConsultaFrutasYHortalizas.aspx"
        self.driver = None
        self.wait = None

    def setup_driver(self):
        """Configura el WebDriver de Selenium"""
        try:
            if IN_COLAB:
                print("🔧 Configurando ChromeDriver para Colab...")

                # Instalar Chrome
                !apt-get update > /dev/null 2>&1
                !apt-get install -y chromium-chromedriver > /dev/null 2>&1

                chrome_options = Options()
                chrome_options.add_argument('--headless')
                chrome_options.add_argument('--no-sandbox')
                chrome_options.add_argument('--disable-dev-shm-usage')
                chrome_options.add_argument('--disable-gpu')
                chrome_options.add_argument('--window-size=1920,1080')
                chrome_options.add_argument('--user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

                self.driver = webdriver.Chrome(options=chrome_options)
            else:
                chrome_options = Options()
                chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
                self.driver = webdriver.Chrome(options=chrome_options)

            self.wait = WebDriverWait(self.driver, 30)
            print("✅ ChromeDriver configurado correctamente")
            return True

        except Exception as e:
            logger.error(f"❌ Error configurando ChromeDriver: {e}")
            return False

    def navigate_to_form(self):
        """Navega al formulario principal - función reutilizable"""
        try:
            print("🔄 Navegando al formulario...")

            # Cargar la página principal
            self.driver.get(self.base_url)

            # Esperar a que la página cargue completamente
            time.sleep(10)

            # Cambiar al iframe que contiene el formulario
            iframes = self.driver.find_elements(By.TAG_NAME, "iframe")
            if not iframes:
                print("❌ No se encontraron iframes")
                return False

            self.driver.switch_to.frame(iframes[0])
            time.sleep(3)

            # Verificar que estamos en el formulario correcto
            test_elements = ["ddlProducto", "txtFechaInicio", "txtFechaFinal"]
            for element_id in test_elements:
                try:
                    element = self.driver.find_element(By.ID, element_id)
                    if element:
                        print(f"✅ Formulario cargado correctamente - Elemento encontrado: {element_id}")
                except:
                    print(f"❌ No se pudo encontrar {element_id} en el formulario")
                    return False

            return True

        except Exception as e:
            logger.error(f"❌ Error navegando al formulario: {e}")
            return False

    def get_initial_page(self):
        """Método para compatibilidad - llama a navigate_to_form"""
        return self.navigate_to_form()

    def switch_to_content_frame(self):
        """Cambia al iframe que contiene el contenido del formulario"""
        try:
            print("🖼️ Buscando iframe de contenido...")

            # Esperar a que el iframe esté disponible
            time.sleep(5)

            # Buscar todos los iframes
            iframes = self.driver.find_elements(By.TAG_NAME, "iframe")
            print(f"📊 Se encontraron {len(iframes)} iframes")

            if not iframes:
                print("❌ No se encontraron iframes")
                return False

            # Usar el primer iframe (generalmente es el correcto)
            try:
                print("🔄 Intentando cambiar al iframe...")
                self.driver.switch_to.frame(iframes[0])

                # Esperar a que el contenido del iframe cargue
                time.sleep(3)
                return True

            except Exception as e:
                print(f"❌ Error cambiando al iframe: {e}")
                return False

        except Exception as e:
            logger.error(f"❌ Error cambiando al iframe: {e}")
            return False

    def find_element_with_retry(self, selectors, by_type=By.ID, description=""):
        """Busca un elemento con múltiples selectores y reintentos"""
        for selector in selectors:
            try:
                if by_type == By.XPATH:
                    element = self.driver.find_element(By.XPATH, selector)
                else:
                    element = self.driver.find_element(By.ID, selector)

                if element.is_displayed():
                    print(f"✅ {description} encontrado con selector: {selector}")
                    return element
            except Exception as e:
                continue

        print(f"❌ No se pudo encontrar {description} con ningún selector: {selectors}")
        return None

    def set_fechas(self, fecha_inicio, fecha_fin):
        """Establece las fechas de consulta"""
        try:
            print("📅 Configurando fechas de consulta...")

            # Usar los selectores correctos que encontramos en el debug
            fecha_inicio_input = self.driver.find_element(By.ID, "txtFechaInicio")
            fecha_fin_input = self.driver.find_element(By.ID, "txtFechaFinal")

            # Limpiar y establecer fecha inicial
            fecha_inicio_input.clear()
            fecha_inicio_input.send_keys(fecha_inicio.strftime('%d/%m/%Y'))
            print(f"📅 Fecha inicial establecida: {fecha_inicio.strftime('%d/%m/%Y')}")

            # Limpiar y establecer fecha final
            fecha_fin_input.clear()
            fecha_fin_input.send_keys(fecha_fin.strftime('%d/%m/%Y'))
            print(f"📅 Fecha final establecida: {fecha_fin.strftime('%d/%m/%Y')}")

            return True

        except Exception as e:
            logger.error(f"❌ Error estableciendo fechas: {e}")
            return False

    def get_productos(self):
        """Extrae la lista de productos del dropdown"""
        try:
            print("📋 Extrayendo lista de productos desde el iframe...")

            # Lista de selectores posibles para el dropdown de productos
            product_selectors = [
                "ctl00_ContentPlaceHolder1_ddlProducto",
                "ddlProducto",
                "ContentPlaceHolder1_ddlProducto"
            ]

            xpath_selectors = [
                "//select[contains(@id, 'Producto')]",
                "//select[contains(@name, 'Producto')]",
                "//select[@id='ctl00_ContentPlaceHolder1_ddlProducto']"
            ]

            # Buscar por ID primero
            dropdown_element = self.find_element_with_retry(product_selectors, By.ID, "dropdown de productos")

            # Si no se encuentra por ID, buscar por XPath
            if not dropdown_element:
                dropdown_element = self.find_element_with_retry(xpath_selectors, By.XPATH, "dropdown de productos")

            if not dropdown_element:
                # Último intento: buscar todos los selects
                print("🔍 Buscando todos los elementos select en el iframe...")
                all_selects = self.driver.find_elements(By.TAG_NAME, "select")
                print(f"📊 Encontrados {len(all_selects)} elementos select")

                for select in all_selects:
                    select_id = select.get_attribute('id') or ''
                    select_name = select.get_attribute('name') or ''
                    print(f"   Select: ID='{select_id}', Name='{select_name}'")

                    if 'producto' in select_id.lower() or 'producto' in select_name.lower():
                        dropdown_element = select
                        break

            if not dropdown_element:
                print("❌ No se pudo encontrar el dropdown de productos en el iframe")
                return []

            # Crear objeto Select y obtener opciones
            select_productos = Select(dropdown_element)
            options = select_productos.options

            print(f"📊 Se encontraron {len(options)} opciones en el dropdown")

            productos = []
            for i, option in enumerate(options):
                value = option.get_attribute("value")
                text = option.text.strip()

                # Solo incluir opciones con valor y que no sea "Todos" o "Seleccione"
                if value and value != "" and value != "-1" and text and text != "Seleccione" and text != "Todos":
                    productos.append({
                        'id': value,
                        'nombre': text,
                        'index': i
                    })

                    # Mostrar los primeros 5 productos para verificación
                    if i < 5:
                        print(f"   {i+1:2d}. {text} (valor: {value})")

            if len(options) > 5:
                print(f"   ... y {len(options) - 5} productos más")

            print(f"✅ Se encontraron {len(productos)} productos válidos")
            return productos

        except Exception as e:
            logger.error(f"❌ Error extrayendo productos: {e}")
            return []

    def get_tipos_precio(self):
        """Obtiene los tipos de precio disponibles"""
        try:
            print("💰 Buscando tipos de precio disponibles...")

            # Buscar dropdown de tipo de precio
            precio_selectors = [
                "ctl00_ContentPlaceHolder1_ddlTipoPrecio",
                "ddlTipoPrecio",
                "TipoPrecio",
                "//select[contains(@id, 'Precio')]",
                "//select[contains(@name, 'Precio')]"
            ]

            precio_dropdown = None
            for selector in precio_selectors:
                try:
                    if selector.startswith("//"):
                        precio_dropdown = self.driver.find_element(By.XPATH, selector)
                    else:
                        precio_dropdown = self.driver.find_element(By.ID, selector)

                    if precio_dropdown:
                        print(f"✅ Dropdown de tipo de precio encontrado: {selector}")
                        break
                except:
                    continue

            if not precio_dropdown:
                print("❌ No se pudo encontrar el dropdown de tipo de precio")
                return []

            # Obtener opciones del dropdown
            select_precio = Select(precio_dropdown)
            options = select_precio.options

            print(f"📊 Opciones de tipo de precio encontradas: {len(options)}")

            tipos_precio = []
            for i, option in enumerate(options):
                value = option.get_attribute("value")
                text = option.text.strip()

                if value and value != "" and text:
                    tipos_precio.append({
                        'id': value,
                        'nombre': text,
                        'element_id': value
                    })
                    print(f"   {i+1}. {text} (valor: {value})")

            # Si no encontramos opciones, usar valores por defecto
            if not tipos_precio:
                print("⚠️ No se encontraron opciones, usando valores por defecto")
                tipos_precio = [
                    {'id': '0', 'nombre': 'Presentación Comercial (encuestado)', 'element_id': '0'},
                    {'id': '1', 'nombre': 'por kilogramo (calculado)', 'element_id': '1'}
                ]

            return tipos_precio

        except Exception as e:
            logger.error(f"❌ Error obteniendo tipos de precio: {e}")
            return [
                {'id': '0', 'nombre': 'Presentación Comercial (encuestado)', 'element_id': '0'},
                {'id': '1', 'nombre': 'por kilogramo (calculado)', 'element_id': '1'}
            ]

    def set_todos_origenes_destinos(self):
        """Selecciona 'Todos' en origen y destino"""
        try:
            # Buscar dropdown de origen
            origen_selectors = [
                "ctl00_ContentPlaceHolder1_ddlOrigen",
                "ddlOrigen"
            ]

            origen_element = self.find_element_with_retry(origen_selectors, By.ID, "dropdown origen")

            if origen_element:
                select_origen = Select(origen_element)

                # Intentar diferentes valores para "Todos"
                todos_values = ["-1", "0", ""]
                for value in todos_values:
                    try:
                        select_origen.select_by_value(value)
                        print(f"📍 Origen configurado: Todos (valor: {value})")
                        break
                    except:
                        continue
                else:
                    # Si no funciona por value, intentar por texto
                    try:
                        select_origen.select_by_visible_text("Todos")
                        print("📍 Origen configurado: Todos (por texto)")
                    except:
                        print("❌ No se pudo seleccionar 'Todos' en origen")
                        return False
            else:
                return False

            # Buscar dropdown de destino
            destino_selectors = [
                "ctl00_ContentPlaceHolder1_ddlDestino",
                "ddlDestino"
            ]

            destino_element = self.find_element_with_retry(destino_selectors, By.ID, "dropdown destino")

            if destino_element:
                select_destino = Select(destino_element)

                # Intentar diferentes valores para "Todos"
                todos_values = ["-1", "0", ""]
                for value in todos_values:
                    try:
                        select_destino.select_by_value(value)
                        print(f"📍 Destino configurado: Todos (valor: {value})")
                        break
                    except:
                        continue
                else:
                    # Si no funciona por value, intentar por texto
                    try:
                        select_destino.select_by_visible_text("Todos")
                        print("📍 Destino configurado: Todos (por texto)")
                    except:
                        print("❌ No se pudo seleccionar 'Todos' en destino")
                        return False

                return True
            else:
                return False

        except Exception as e:
            logger.error(f"❌ Error configurando orígenes/destinos: {e}")
            return False

    def seleccionar_producto(self, producto_id):
        """Selecciona un producto específico"""
        try:
            product_selectors = [
                "ctl00_ContentPlaceHolder1_ddlProducto",
                "ddlProducto"
            ]

            dropdown_element = self.find_element_with_retry(product_selectors, By.ID, "dropdown productos")

            if not dropdown_element:
                return False

            select_productos = Select(dropdown_element)
            select_productos.select_by_value(producto_id)

            # Esperar a que la página procese la selección
            time.sleep(3)
            return True

        except Exception as e:
            logger.error(f"❌ Error seleccionando producto {producto_id}: {e}")
            return False

    def seleccionar_tipo_precio(self, tipo_precio_element_id, tipo_precio_nombre):
        """Selecciona el tipo de precio"""
        try:
            print(f"💰 Intentando seleccionar tipo de precio: {tipo_precio_nombre}")

            # Buscar dropdown de tipo de precio
            precio_selectors = [
                "ctl00_ContentPlaceHolder1_ddlTipoPrecio",
                "ddlTipoPrecio",
                "TipoPrecio",
                "//select[contains(@id, 'Precio')]",
                "//select[contains(@name, 'Precio')]"
            ]

            precio_dropdown = None
            for selector in precio_selectors:
                try:
                    if selector.startswith("//"):
                        precio_dropdown = self.driver.find_element(By.XPATH, selector)
                    else:
                        precio_dropdown = self.driver.find_element(By.ID, selector)

                    if precio_dropdown:
                        print(f"✅ Dropdown de tipo de precio encontrado: {selector}")
                        break
                except:
                    continue

            if not precio_dropdown:
                print("❌ No se pudo encontrar el dropdown de tipo de precio")
                return False

            # Seleccionar la opción en el dropdown
            select_precio = Select(precio_dropdown)

            # Intentar seleccionar por valor
            try:
                select_precio.select_by_value(tipo_precio_element_id)
                print(f"✅ Tipo de precio seleccionado por valor: {tipo_precio_nombre}")
                return True
            except:
                # Si no funciona por valor, intentar por texto
                try:
                    select_precio.select_by_visible_text(tipo_precio_nombre)
                    print(f"✅ Tipo de precio seleccionado por texto: {tipo_precio_nombre}")
                    return True
                except:
                    print(f"❌ No se pudo seleccionar el tipo de precio: {tipo_precio_nombre}")
                    return False

        except Exception as e:
            logger.error(f"❌ Error seleccionando tipo de precio: {e}")
            return False

    def hacer_consulta(self):
        """Ejecuta la consulta y espera los resultados"""
        try:
            # Buscar botón de búsqueda
            boton_selectors = [
                "ctl00_ContentPlaceHolder1_btnBuscar",
                "btnBuscar",
                "Buscar",
                "//input[@type='submit' and contains(@value, 'Buscar')]",
                "//input[@type='button' and contains(@value, 'Buscar')]",
                "//button[contains(text(), 'Buscar')]",
                "//input[contains(@id, 'Buscar')]",
                "//*[contains(text(), 'Buscar') and (self::input or self::button)]"
            ]

            boton_buscar = None
            for selector in boton_selectors:
                try:
                    if selector.startswith("//"):
                        elements = self.driver.find_elements(By.XPATH, selector)
                        if elements:
                            boton_buscar = elements[0]
                            print(f"✅ Botón 'Buscar' encontrado con XPath: {selector}")
                            break
                    else:
                        boton_buscar = self.driver.find_element(By.ID, selector)
                        print(f"✅ Botón 'Buscar' encontrado con ID: {selector}")
                        break
                except:
                    continue

            if not boton_buscar:
                print("❌ No se pudo encontrar el botón 'Buscar'")
                return False

            # Desplazarse al botón si es necesario
            self.driver.execute_script("arguments[0].scrollIntoView(true);", boton_buscar)
            time.sleep(1)

            boton_buscar.click()
            print("🔄 Ejecutando búsqueda...")

            # Esperar a que los resultados se carguen
            time.sleep(10)

            # Verificar si hay resultados
            page_source = self.driver.page_source
            if "No se encontraron registros" in page_source:
                print("ℹ️ No se encontraron registros para esta consulta")
                return True
            elif "gvResultados" in page_source:
                print("✅ Resultados cargados correctamente")
                return True
            else:
                print("⚠️ No se pudo determinar el estado de la consulta")
                return True

        except Exception as e:
            logger.error(f"❌ Error ejecutando consulta: {e}")
            return False

    def extraer_datos_tabla(self, producto_id, tipo_precio_id, producto_nombre, tipo_precio_nombre):
        """Extrae los datos de la tabla de resultados"""
        try:
            datos = []

            # Verificar si hay tabla de resultados
            if "No se encontraron registros" in self.driver.page_source:
                print("ℹ️ No hay registros para extraer")
                return datos

            # Buscar tabla de resultados
            tabla_selectors = [
                "ctl00_ContentPlaceHolder1_gvResultados",
                "gvResultados",
                "//table[contains(@id, 'Resultados')]",
                "//table[contains(@class, 'grid')]"
            ]

            tabla_element = None
            for selector in tabla_selectors:
                try:
                    if selector.startswith("//"):
                        tabla_element = self.driver.find_element(By.XPATH, selector)
                    else:
                        tabla_element = self.driver.find_element(By.ID, selector)

                    if tabla_element:
                        break
                except:
                    continue

            if not tabla_element:
                print("ℹ️ No se encontró tabla de resultados")
                return datos

            filas = tabla_element.find_elements(By.TAG_NAME, "tr")
            print(f"📊 Encontradas {len(filas)} filas en la tabla")

            # VERIFICACIÓN MEJORADA PARA IDENTIFICAR FILAS DE ENCABEZADO
            start_index = 0

            # Verificar si la primera fila es encabezado (contiene th o textos específicos)
            if len(filas) > 0:
                primera_fila = filas[0]
                # Verificar si tiene celdas de encabezado (th)
                th_cells = primera_fila.find_elements(By.TAG_NAME, "th")
                if th_cells:
                    start_index = 1
                    print("✅ Se identificó fila de encabezado (th)")
                else:
                    # Verificar por contenido típico de encabezados
                    td_cells = primera_fila.find_elements(By.TAG_NAME, "td")
                    if td_cells:
                        texto_primera_fila = ' '.join([celda.text.strip() for celda in td_cells])
                        palabras_encabezado = ['Fecha', 'Presentación', 'Origen', 'Destino', 'Precio', 'Mínimo', 'Máximo', 'Frecuente', 'Observaciones']

                        if any(palabra in texto_primera_fila for palabra in palabras_encabezado):
                            start_index = 1
                            print("✅ Se identificó fila de encabezado por contenido")

            print(f"📊 Procesando desde fila {start_index + 1} de {len(filas)}")

            # Procesar filas de datos
            for i in range(start_index, len(filas)):
                try:
                    fila = filas[i]
                    celdas = fila.find_elements(By.TAG_NAME, "td")

                    # VERIFICACIÓN ADICIONAL: Saltar filas que parezcan encabezados
                    if len(celdas) > 0:
                        texto_fila = ' '.join([celda.text.strip() for celda in celdas])
                        palabras_encabezado = ['Fecha', 'Presentación', 'Origen', 'Destino', 'Precio', 'Mínimo', 'Máximo', 'Frecuente', 'Observaciones']

                        # Si más del 50% de las palabras de encabezado están en la fila, saltarla
                        palabras_encontradas = sum(1 for palabra in palabras_encabezado if palabra in texto_fila)
                        if palabras_encontradas >= 3:  # Ajusta este número según sea necesario
                            print(f"⚠️ Saltando fila {i+1} (parece encabezado): {texto_fila[:100]}...")
                            continue

                    if len(celdas) >= 6:  # Mínimo 6 columnas esperadas
                        # VERIFICACIÓN DE DATOS VÁLIDOS
                        fecha = celdas[0].text.strip() if len(celdas) > 0 else ''
                        presentacion = celdas[1].text.strip() if len(celdas) > 1 else ''
                        origen = celdas[2].text.strip() if len(celdas) > 2 else ''

                        # Si la fecha no tiene formato de fecha, probablemente sea encabezado
                        if not re.match(r'\d{1,2}/\d{1,2}/\d{4}', fecha) and any(palabra in fecha for palabra in ['Fecha', 'Día']):
                            print(f"⚠️ Saltando fila {i+1} (formato de fecha inválido): {fecha}")
                            continue

                        dato = {
                            'producto_id': producto_id,
                            'producto_nombre': producto_nombre,
                            'tipo_precio_id': tipo_precio_id,
                            'tipo_precio_nombre': tipo_precio_nombre,
                            'fecha': fecha,
                            'presentacion': presentacion,
                            'origen': origen,
                            'destino': celdas[3].text.strip() if len(celdas) > 3 else '',
                            'precio_min': self.limpiar_precio(celdas[4].text.strip() if len(celdas) > 4 else ''),
                            'precio_max': self.limpiar_precio(celdas[5].text.strip() if len(celdas) > 5 else ''),
                            'precio_frec': self.limpiar_precio(celdas[6].text.strip() if len(celdas) > 6 else ''),
                            'observaciones': celdas[7].text.strip() if len(celdas) > 7 else '',
                            'fecha_consulta': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                        }

                        # Solo agregar si tiene datos válidos
                        if dato['fecha'] and dato['presentacion']:
                            datos.append(dato)
                        else:
                            print(f"⚠️ Saltando fila {i+1} (datos insuficientes)")

                except Exception as e:
                    print(f"⚠️ Error procesando fila {i+1}: {e}")
                    continue

            print(f"✅ Se extrajeron {len(datos)} registros válidos")
            return datos

        except Exception as e:
            logger.error(f"❌ Error extrayendo tabla: {e}")
            return []

    def limpiar_precio(self, precio_str):
        """Limpia y convierte el precio a float"""
        if not precio_str or precio_str.strip() == '':
            return None

        # Remover caracteres no numéricos excepto punto decimal
        precio_limpio = re.sub(r'[^\d.]', '', precio_str)

        try:
            return float(precio_limpio) if precio_limpio else None
        except ValueError:
            return None

    def guardar_datos(self, datos, directorio_salida, archivo_nombre):
        """Guarda los datos en archivos CSV"""
        if not datos:
            logger.warning("⚠️ No hay datos para guardar")
            return None

        os.makedirs(directorio_salida, exist_ok=True)

        # Crear DataFrame
        df = pd.DataFrame(datos)

        # Nombre del archivo
        archivo_csv = os.path.join(directorio_salida, f"{archivo_nombre}.csv")

        # Guardar en CSV
        df.to_csv(archivo_csv, index=False, encoding='utf-8-sig')
        logger.info(f"💾 Datos guardados en: {archivo_csv}")

        return archivo_csv

    def reset_form(self):
        """Vuelve al formulario principal después de una consulta"""
        try:
            print("🔄 Restableciendo formulario...")

            # Volver al contexto principal
            self.driver.switch_to.default_content()

            # Navegar de nuevo al formulario
            return self.navigate_to_form()

        except Exception as e:
            logger.error(f"❌ Error restableciendo formulario: {e}")
            return False

    def close(self):
        """Cierra el navegador"""
        if self.driver:
            # Volver al contexto principal antes de cerrar
            try:
                self.driver.switch_to.default_content()
            except:
                pass
            self.driver.quit()
            print("🔒 Navegador cerrado")

def configurar_directorio_colab():
    """Configura Google Drive para Colab"""
    if IN_COLAB:
        print("📁 Montando Google Drive...")
        drive.mount('/content/drive')

        # Crear directorio base
        directorio_base = "/content/drive/MyDrive/FRUTAS_HORTALIZAS"
        os.makedirs(directorio_base, exist_ok=True)

        print(f"✅ Directorio configurado: {directorio_base}")
        return directorio_base
    else:
        # Directorio local para pruebas
        directorio_local = "FRUTAS_HORTALIZAS"
        os.makedirs(directorio_local, exist_ok=True)
        print(f"📂 Directorio local: {directorio_local}")
        return directorio_local

def main():
    """Función principal"""
    print("=" * 70)
    print("🌱 EXTRACTOR DE DATOS SNIIM - FRUTAS Y HORTALIZAS")
    print("📍 Selenium Edition - VERSIÓN FINAL COMPLETA")
    print("=" * 70)

    # Configurar directorio de salida
    directorio_salida = configurar_directorio_colab()

    # Inicializar extractor con Selenium
    extractor = SNIIMExtractorSelenium()

    # Configurar el driver
    if not extractor.setup_driver():
        print("❌ No se pudo configurar Selenium. Terminando ejecución.")
        return

    try:
        # Obtener página inicial
        print("\n🔗 Navegando a la página del SNIIM...")
        if not extractor.get_initial_page():
            print("❌ No se pudo cargar la página inicial.")
            return

        # Obtener lista de productos
        print("\n📋 Extrayendo lista de productos...")
        productos = extractor.get_productos()

        if not productos:
            print("❌ No se encontraron productos disponibles.")
            return

        # Obtener tipos de precio
        print("\n💰 Detectando tipos de precio disponibles...")
        tipos_precio = extractor.get_tipos_precio()

        if not tipos_precio:
            print("❌ No se encontraron tipos de precio.")
            return

        # Configurar fechas (últimos 7 días para prueba)
        fecha_fin = datetime.now() - timedelta(days=1)  # Ayer
        fecha_inicio = fecha_fin - timedelta(days=7)   # Últimos 7 días

        print(f"\n📅 Rango de fechas: {fecha_inicio.strftime('%d/%m/%Y')} - {fecha_fin.strftime('%d/%m/%Y')}")
        print(f"🍎 Productos disponibles: {len(productos)}")
        print(f"💰 Tipos de precio: {len(tipos_precio)}")

        # Procesar TODOS los productos
        productos_a_procesar = productos
        print(f"🔍 Procesando TODOS los {len(productos_a_procesar)} productos...")

        # Recolectar todos los datos
        print(f"\n🎬 INICIANDO EXTRACCIÓN MASIVA...")

        todos_los_datos = []
        productos_exitosos = 0
        productos_con_errores = 0

        for i, producto in enumerate(productos_a_procesar, 1):
            print(f"\n{'='*60}")
            print(f"🍎 [{i}/{len(productos_a_procesar)}] PROCESANDO: {producto['nombre']}")
            print(f"{'='*60}")

            producto_con_datos = False

            for j, tipo_precio in enumerate(tipos_precio, 1):
                print(f"\n💰 [{j}/{len(tipos_precio)}] Tipo: {tipo_precio['nombre']}")

                try:
                    # Configurar fechas y origen/destino para cada consulta
                    if not extractor.set_fechas(fecha_inicio, fecha_fin):
                        print("❌ Error configurando fechas, continuando...")
                        continue

                    if not extractor.set_todos_origenes_destinos():
                        print("❌ Error configurando orígenes/destinos, continuando...")
                        continue

                    # Seleccionar producto
                    if not extractor.seleccionar_producto(producto['id']):
                        print("❌ Error seleccionando producto, continuando...")
                        continue

                    # Seleccionar tipo de precio
                    if not extractor.seleccionar_tipo_precio(tipo_precio['element_id'], tipo_precio['nombre']):
                        print("❌ Error seleccionando tipo de precio, continuando...")
                        continue

                    # Ejecutar consulta
                    if not extractor.hacer_consulta():
                        print("❌ Error en consulta, continuando...")
                        continue

                    # Extraer datos
                    datos_producto = extractor.extraer_datos_tabla(
                        producto['id'],
                        tipo_precio['id'],
                        producto['nombre'],
                        tipo_precio['nombre']
                    )

                    todos_los_datos.extend(datos_producto)

                    if datos_producto:
                        producto_con_datos = True
                        print(f"✅ {len(datos_producto)} registros obtenidos")
                    else:
                        print("ℹ️ Sin registros para este tipo de precio")

                    # Restablecer formulario después de cada consulta
                    if not extractor.reset_form():
                        print("❌ No se pudo restablecer el formulario, intentando continuar...")
                        # Intentar una recuperación más agresiva
                        extractor.close()
                        time.sleep(5)
                        if not extractor.setup_driver() or not extractor.get_initial_page():
                            print("❌ No se pudo recuperar la sesión, terminando ejecución.")
                            break

                except Exception as e:
                    print(f"❌ Error inesperado procesando {producto['nombre']} - {tipo_precio['nombre']}: {e}")
                    # Intentar recuperar el formulario
                    if not extractor.reset_form():
                        extractor.close()
                        time.sleep(5)
                        if not extractor.setup_driver() or not extractor.get_initial_page():
                            print("❌ No se pudo recuperar la sesión, terminando ejecución.")
                            break

                # Esperar entre consultas
                time.sleep(2)

            # Resumen del producto
            if producto_con_datos:
                productos_exitosos += 1
                print(f"✅ {producto['nombre']}: PROCESADO CON ÉXITO")
            else:
                productos_con_errores += 1
                print(f"⚠️ {producto['nombre']}: NO SE OBTUVIERON DATOS")

            # Guardar datos parciales cada 5 productos
            if i % 5 == 0 and todos_los_datos:
                timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                archivo_parcial = extractor.guardar_datos(
                    todos_los_datos,
                    directorio_salida,
                    f"sniim_parcial_{i}_{timestamp}"
                )
                print(f"💾 Guardado parcial ({i} productos): {archivo_parcial}")

        # Guardar datos finales
        if todos_los_datos:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            archivo_final = extractor.guardar_datos(
                todos_los_datos,
                directorio_salida,
                f"sniim_completo_{timestamp}"
            )

            print(f"\n{'='*80}")
            print("🎉 EXTRACCIÓN COMPLETADA")
            print(f"{'='*80}")
            print(f"📊 TOTAL DE REGISTROS: {len(todos_los_datos):,}")
            print(f"🍎 PRODUCTOS EXITOSOS: {productos_exitosos}/{len(productos_a_procesar)}")
            print(f"⚠️ PRODUCTOS SIN DATOS: {productos_con_errores}/{len(productos_a_procesar)}")
            print(f"💰 TIPOS DE PRECIO: {len(tipos_precio)}")
            print(f"📅 PERÍODO: {fecha_inicio.strftime('%d/%m/%Y')} - {fecha_fin.strftime('%d/%m/%Y')}")
            print(f"💾 ARCHIVO GUARDADO: {archivo_final}")

        else:
            print("\n❌ No se extrajeron datos.")

    except Exception as e:
        logger.error(f"❌ Error en la ejecución principal: {str(e)}")
        import traceback
        traceback.print_exc()

        # Intentar guardar datos recolectados hasta el momento
        if 'todos_los_datos' in locals() and todos_los_datos:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            archivo_error = extractor.guardar_datos(
                todos_los_datos,
                directorio_salida,
                f"sniim_error_{timestamp}"
            )
            print(f"💾 Datos guardados hasta el error: {archivo_error}")

    finally:
        extractor.close()

if __name__ == "__main__":
    # Instalar Selenium si es necesario en Colab
    if IN_COLAB:
        print("📦 Instalando Selenium...")
        !pip install selenium -q

    main()

✅ Ejecutando en Google Colab
📦 Instalando Selenium...
🌱 EXTRACTOR DE DATOS SNIIM - FRUTAS Y HORTALIZAS
📍 Selenium Edition - VERSIÓN FINAL COMPLETA
📁 Montando Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Directorio configurado: /content/drive/MyDrive/FRUTAS_HORTALIZAS
🔧 Configurando ChromeDriver para Colab...
✅ ChromeDriver configurado correctamente

🔗 Navegando a la página del SNIIM...
🔄 Navegando al formulario...
✅ Formulario cargado correctamente - Elemento encontrado: ddlProducto
✅ Formulario cargado correctamente - Elemento encontrado: txtFechaInicio
✅ Formulario cargado correctamente - Elemento encontrado: txtFechaFinal

📋 Extrayendo lista de productos...
📋 Extrayendo lista de productos desde el iframe...
✅ dropdown de productos encontrado con selector: ddlProducto
📊 Se encontraron 223 opciones en el dropdown
    2. Acelga - Primera (valor: 31)
    3. Aguacate Criollo - Primera (v

🔒 Navegador cerrado


KeyboardInterrupt: 